# Basic data cleaning with NumPy and pandas
Clean `youtube_music_history_expanded.csv` and save the cleaned CSV in the repository’s data folder. Only basic pandas and NumPy operations are used for cleaning.

Missing play counts become **0** (assuming a blank means no recorded plays). Unknown text, dates, ranks, years, views, and durations stay **null** (`np.nan` in memory). A missing measurement is not necessarily zero. Completely empty columns are deleted after whitespace is cleaned. Rows and duplicate records are preserved.


In [ ]:
import numpy as np
import pandas as pd

input_file = "../data/youtube_music_history_expanded.csv"
output_file = "../data/youtube_music_history_cleaned.csv"
df = pd.read_csv(input_file)
print("Original shape:", df.shape)
df.head()


## 1. Check missing values and column types

In [ ]:
original_shape = df.shape
missing_before = df.isna().sum()
print(df.dtypes)
missing_before


## 2. Trim spaces and turn blank text into null
Whitespace-only cells also count as empty. Existing CSV missing values remain null.

In [ ]:
df.columns = df.columns.str.strip()

for column in df.select_dtypes(include=["object", "str"]).columns:
    df[column] = df[column].str.strip()
    df[column] = df[column].replace("", np.nan)

df.head()


## 3. Delete fully empty columns
A column is removed only when every value is missing.

In [ ]:
empty_columns = df.columns[df.isna().all()].tolist()
df = df.dropna(axis=1, how="all")
print("Removed empty columns:", empty_columns)
print("Shape after removal:", df.shape)


## 4. Fill missing values where appropriate
Only missing play counts are filled with zero. Keep unknown values in other columns as null. Edit the list below if a different dataset has other columns where zero has a valid meaning.

In [ ]:
zero_fill_columns = ["History Play Count"]

for column in zero_fill_columns:
    if column in df.columns:
        df[column] = df[column].fillna(0)

# All remaining missing values represent unknown information.
df = df.fillna(np.nan)
df.head()


## 5. Review and save
The CSV writes remaining missing values as the literal `null`. pandas recognizes this marker as missing when reading the file back.

In [ ]:
missing_summary = pd.DataFrame({
    "Missing before": missing_before,
    "Missing after": df.isna().sum()
})
print("Original shape:", original_shape)
print("Cleaned shape:", df.shape)
print(missing_summary)

df.to_csv(output_file, index=False, na_rep="null")
print("Saved:", output_file)
